In [1]:
import cuml

from cuml.ensemble import RandomForestRegressor

from sklearn import ensemble
from sklearn.ensemble import ExtraTreesRegressor

import sklearn

import matplotlib.pyplot as plt

import xgboost as xgb
from lightgbm import LGBMRegressor

import torch

import numpy as np
import cupy as cp

import pandas as pd
from pathlib import Path
import gc

from sklearn.metrics import r2_score

import time

In [2]:
randomf = True
extrat = True
xgboost = True
lgboostm = True

In [3]:
modelsLib = {
    "rf": 1,
    "rfGPU": 1,
    "et": 1,
    "etGPU": 1,
    "xgb": 1,
    "xgbGPU": 1,
    "lgbm": 1,
    "lgbmGPU": 1
}

In [4]:
ET = {
    "model": ExtraTreesRegressor,
    "n_estimators": [50, 50, 50],
    "max_depth": [21, 21, 1],
    "min_samples_leaf": [4, 4, 2],
    "max_features": [1.0, 1.0, 0.1],
    "results": []
}

X = {
    "model": xgb,
    "n_estimators": [500, 500, 50],
    "max_depth": [15, 15, 1],
    "min_child_weight": [3, 3, 1],
    "results": []
}

LGBM = {
    "model": LGBMRegressor,
    "n_estimators": [450, 450, 50],
    "min_child_samples": [15, 15, 10],
    "num_leaves": [256, 256, 2],
    "colsample_bytree": [0.7, 0.7, 0.1],
    "results": [],
    "trials": []
}

In [5]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/shadows-detection-dataset/trees-gt-shadows.csv")

#making sure
datasetPath.is_file()

True

In [6]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0


In [7]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.26, 0.26, 0.26, ..., 0.26, 0.26, 0.26],
      shape=(332640,), dtype=float32)

In [8]:
inputs = dataset.drop("grviOut", axis=1)
inputs = inputs.drop("sunRoll", axis=1)
inputs = inputs.drop("camRoll", axis=1)

inputs.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunPitch,sunYaw,camPitch,camYaw,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,-71.945084,19.489212,-19.35,-0.0,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,-71.945084,19.489212,-19.35,-0.0,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,-71.945084,19.489212,-19.35,-0.0,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,-71.945084,19.489212,-19.35,-0.0,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,-71.945084,19.489212,-19.35,-0.0,0.0


In [9]:
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 2.4851234e-01,  1.4691126e-04,  2.0387112e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       [ 2.0935635e-01,  2.5367978e-04,  3.4656093e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       [ 2.2862221e-01,  2.2864803e-04,  3.1710259e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       ...,
       [ 8.0871526e-03,  3.2670033e-01,  3.1690043e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00],
       [ 1.2527064e-02,  2.7300212e-01,  2.6233262e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00],
       [ 2.2494195e-02,  2.6528352e-01,  2.5932705e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00]],
      shape=(332640, 11), dtype=float32)

In [10]:
n_total = len(inputs)

n_train = int(0.9*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]

In [11]:
mask_high = val_inputs[:, 10] == 0.0
mask_low = ~mask_high

val_inputs_high = val_inputs[mask_high]
val_labels_high = val_labels[mask_high]

val_inputs_low = val_inputs[mask_low]
val_labels_low = val_labels[mask_low]

val_inputs = np.delete(val_inputs, 10, axis=1)
val_inputs_high = np.delete(val_inputs_high, 10, axis=1)
val_inputs_low = np.delete(val_inputs_low, 10, axis=1)

train_inputs = np.delete(train_inputs, 10, axis=1)

In [12]:
ne = ET["n_estimators"][0]
md = ET["max_depth"][0]
nb = ET["min_samples_leaf"][0]
mf = ET["max_features"][0]

modelET = ET["model"](n_estimators=ne, max_depth=md, min_samples_leaf=nb, max_features=mf)
modelET.fit(train_inputs, train_labels)

y_pred = modelET.predict(val_inputs)
y_pred_shadows = modelET.predict(val_inputs_low)
y_pred_clear = modelET.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels, y_pred))
rmse_shadows = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred_shadows))
rmse_clear = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred_clear))

r2 = r2_score(val_labels, y_pred)
r2_shadows = r2_score(val_labels_low, y_pred_shadows)
r2_clear = r2_score(val_labels_high, y_pred_clear)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())
rel_test_loss_shadows = (rmse_shadows*100) / (val_labels_low.max()-val_labels_low.min())
rel_test_loss_clear = (rmse_clear*100) / (val_labels_high.max()-val_labels_high.min())


print ("overall: RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2 " + str(r2))
print ("shadows: RelRMSE: " + str(float(rel_test_loss_shadows)) + "%, " + "r2 " + str(r2_shadows))
print ("clear: RelRMSE: " + str(float(rel_test_loss_clear)) + "%, " + "r2 " + str(r2_clear))


overall: RelRMSE: 5.915324706053272%, r2 0.920611841436813
shadows: RelRMSE: 6.045013297626757%, r2 0.9071365486649317
clear: RelRMSE: 5.878124720879541%, r2 0.9238292142950144


In [13]:
ne = X["n_estimators"][0]
md = X["max_depth"][0]
mc = X["min_child_weight"][0]

modelXGB = X["model"].XGBRegressor(n_estimators=ne, max_depth=md, min_child_weight=mc, learning_rate=0.05, tree_method="gpu_hist", gpu_id=0)
modelXGB.fit(train_inputs, train_labels)

y_pred = modelXGB.predict(val_inputs)
y_pred_shadows = modelXGB.predict(val_inputs_low)
y_pred_clear = modelXGB.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels, y_pred))
rmse_shadows = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred_shadows))
rmse_clear = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred_clear))

r2 = r2_score(val_labels, y_pred)
r2_shadows = r2_score(val_labels_low, y_pred_shadows)
r2_clear = r2_score(val_labels_high, y_pred_clear)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())
rel_test_loss_shadows = (rmse_shadows*100) / (val_labels_low.max()-val_labels_low.min())
rel_test_loss_clear = (rmse_clear*100) / (val_labels_high.max()-val_labels_high.min())


print ("overall: RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2 " + str(r2))
print ("shadows: RelRMSE: " + str(float(rel_test_loss_shadows)) + "%, " + "r2 " + str(r2_shadows))
print ("clear: RelRMSE: " + str(float(rel_test_loss_clear)) + "%, " + "r2 " + str(r2_clear))

overall: RelRMSE: 5.688085644031235%, r2 0.9265941381454468
shadows: RelRMSE: 5.561751644294581%, r2 0.9213907718658447
clear: RelRMSE: 5.723312010379651%, r2 0.9277886152267456


In [14]:
ne = LGBM["n_estimators"][0]
mcs = LGBM["min_child_samples"][0]
nl = LGBM["num_leaves"][0]
cb = LGBM["colsample_bytree"][0]

modelLGBM = LGBM["model"](n_estimators=ne, min_child_samples=mcs, num_leaves=nl-1, colsample_bytree=cb)
modelLGBM.fit(train_inputs, train_labels)

y_pred = modelLGBM.predict(val_inputs)
y_pred_shadows = modelLGBM.predict(val_inputs_low)
y_pred_clear = modelLGBM.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels, y_pred))
rmse_shadows = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred_shadows))
rmse_clear = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred_clear))

r2 = r2_score(val_labels, y_pred)
r2_shadows = r2_score(val_labels_low, y_pred_shadows)
r2_clear = r2_score(val_labels_high, y_pred_clear)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())
rel_test_loss_shadows = (rmse_shadows*100) / (val_labels_low.max()-val_labels_low.min())
rel_test_loss_clear = (rmse_clear*100) / (val_labels_high.max()-val_labels_high.min())


print ("overall: RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2 " + str(r2))
print ("shadows: RelRMSE: " + str(float(rel_test_loss_shadows)) + "%, " + "r2 " + str(r2_shadows))
print ("clear: RelRMSE: " + str(float(rel_test_loss_clear)) + "%, " + "r2 " + str(r2_clear))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006561 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1635
[LightGBM] [Info] Number of data points in the train set: 299376, number of used features: 10
[LightGBM] [Info] Start training from score 0.305551
overall: RelRMSE: 6.347180559036892%, r2 0.9085970408471515
shadows: RelRMSE: 6.214212547246526%, r2 0.9018653209503517
clear: RelRMSE: 6.384286625305578%, r2 0.9101463765117289
